# 5. Evaluation

## 5.1 Objective
Evaluate the trained XGBoost and Isolation Forest models on the test set using metrics adapted to cybersecurity:
1. **PR-AUC (Precision-Recall AUC):** The most reliable metric for highly imbalanced data.
2. **Confusion Matrix:** To visualize False Positives (FP) and False Negatives (FN).
3. **Asymmetric Cost:** A False Negative (missing an attack) is 10x more expensive than a False Positive. We will find the optimal probability threshold to minimize this business cost.

In [6]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc

# Charger les données de test
X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test.parquet")['Target']

# Charger le modèle XGBoost
xgb_model = joblib.load("../models/xgb_nids.joblib")

# Prédire les probabilités (classe 1 = Attaque)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(f"Données de test chargées : {X_test.shape[0]} lignes.")

Données de test chargées : 19982 lignes.


## 5.2 Standard Evaluation (Threshold = 0.5)

Let's first look at how the model performs with a default 0.5 decision threshold.

In [7]:
# Calcul de la PR-AUC
precision, recall, _ = precision_recall_curve(y_test, y_proba_xgb)
pr_auc = auc(recall, precision)
print(f"PR-AUC Score (XGBoost): {pr_auc:.4f}\n")

# Prédictions avec le seuil par défaut (0.5)
y_pred_default = (y_proba_xgb >= 0.5).astype(int)

print("--- Classification Report (Threshold = 0.5) ---")
print(classification_report(y_test, y_pred_default, target_names=['Benign', 'Attack']))

PR-AUC Score (XGBoost): 1.0000

--- Classification Report (Threshold = 0.5) ---
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     19546
      Attack       1.00      1.00      1.00       436

    accuracy                           1.00     19982
   macro avg       1.00      1.00      1.00     19982
weighted avg       1.00      1.00      1.00     19982



In [8]:
## 5.3 Asymmetric Cost Analysis

In a SOC context, missing an attack (False Negative) is catastrophic. We assume `FN = 10 * FP`. 
Let's calculate the total cost for different thresholds to find the one that minimizes our business cost.

SyntaxError: unterminated string literal (detected at line 4) (3331905723.py, line 4)